In [ ]:
import pandas as pd
import numpy as np
import os
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

# ----------------------------------------------------
# 1. Configuration
# ----------------------------------------------------
DATA_DIR = 'YOUR_DATA_DIRECTORY'  # Replace with your data directory
YEARS = list(range(1957, 2017))
FIRM_CHARS = ['characteristic_mvel1', 'characteristic_bm', 'characteristic_mom12m']
REQUIRED_COLS = ['permno', 'month', 'ret_excess'] + FIRM_CHARS

ESTIMATION_PERIODS = pd.DataFrame({
    'oos_year': range(1987, 2017),
    'training_start': [1957] * 30,
    'training_end': range(1986, 2016)
})[['training_start', 'training_end', 'oos_year']]

# ----------------------------------------------------
# 2. Functions
# ----------------------------------------------------
def load_data(row):
    """Load and split data into training and OOS sets for a given OOS year."""
    train_dfs = []
    oos_dfs = []

    for year in YEARS:
        file_path = os.path.join(DATA_DIR, f'year_{year}.parquet')
        if not os.path.exists(file_path):
            continue

        df_year = pd.read_parquet(file_path, columns=REQUIRED_COLS)
        df_year['month'] = pd.to_datetime(df_year['month'])
        df_year = df_year.dropna()

        if row['training_start'] <= year <= row['training_end']:
            train_dfs.append(df_year)
        elif year == row['oos_year']:
            oos_dfs.append(df_year)

    train_df = pd.concat(train_dfs, ignore_index=True) if train_dfs else pd.DataFrame()
    oos_df = pd.concat(oos_dfs, ignore_index=True) if oos_dfs else pd.DataFrame()
    return train_df, oos_df

def train_model(train_df, oos_df, oos_year):
    """Train OLS model, predict on OOS data, calculate R² measures, and compute short-long portfolio returns."""
    if train_df.empty or oos_df.empty:
        print(f"Skipping OOS year {oos_year} — no train or OOS data")
        return None, None, None

    X_train = train_df[FIRM_CHARS]
    y_train = train_df['ret_excess']
    X_oos = oos_df[FIRM_CHARS]
    y_oos = oos_df['ret_excess']

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_oos_scaled = scaler.transform(X_oos)

    model = LinearRegression()  # Sklearn by default includes an intercept, so there is no need to add it manually
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_oos_scaled)

    # Calculate R² measures
    # Normal R²: 1 - (rss / tss), where tss uses expanding mean from training period
    baseline_mean = train_df['ret_excess'].mean()
    rss = np.sum((y_oos - y_pred) ** 2)
    tss = np.sum((y_oos - baseline_mean) ** 2)
    oos_r2 = 1 - (rss / tss) if tss != 0 else 0

    # Naive R²: 1 - (rss / sum(y_oos^2)), baseline is zero
    y_oos_squared = np.sum(y_oos ** 2)
    naive_oos_r2 = 1 - (rss / y_oos_squared) if y_oos_squared != 0 else 0

    # Short-Long Strategy: Form monthly portfolios based on y_pred
    portfolio_data = oos_df[['month', 'permno', 'ret_excess', 'characteristic_mvel1']].copy()
    portfolio_data['y_pred'] = y_pred
    months = portfolio_data['month'].unique()
    ew_returns = []
    mcap_returns = []

    for month in months:
        monthly_data = portfolio_data[portfolio_data['month'] == month].copy()
        monthly_data = monthly_data.sort_values('y_pred')
        n_stocks = len(monthly_data)
        if n_stocks < 20:  # Ensure enough stocks for deciles
            continue

        # Top and bottom 10% (deciles)
        n_select = max(1, n_stocks // 10)
        long_stocks = monthly_data.iloc[-n_select:]
        short_stocks = monthly_data.iloc[:n_select]

        # Equal-Weighted Returns
        long_ew_return = long_stocks['ret_excess'].mean()
        short_ew_return = short_stocks['ret_excess'].mean()
        ew_portfolio_return = long_ew_return - short_ew_return
        ew_returns.append(ew_portfolio_return)

        # Market-Cap-Weighted Returns
        long_mcap_weights = long_stocks['characteristic_mvel1'] / long_stocks['characteristic_mvel1'].sum()
        short_mcap_weights = short_stocks['characteristic_mvel1'] / short_stocks['characteristic_mvel1'].sum()
        long_mcap_return = (long_stocks['ret_excess'] * long_mcap_weights).sum()
        short_mcap_return = (short_stocks['ret_excess'] * short_mcap_weights).sum()
        mcap_portfolio_return = long_mcap_return - short_mcap_return
        mcap_returns.append(mcap_portfolio_return)

    temp = oos_df[['month', 'permno']].copy()
    temp['y_true'] = y_oos
    temp['y_pred'] = y_pred
    temp['oos_r2'] = oos_r2
    temp['naive_oos_r2'] = naive_oos_r2
    return temp, ew_returns, mcap_returns

def calculate_sharpe_ratio(returns):
    """Calculate annualized Sharpe ratio for portfolio returns."""
    if not returns:
        return np.nan
    returns = np.array(returns)
    mean_monthly = np.mean(returns)
    std_monthly = np.std(returns, ddof=1)
    mean_annualized = mean_monthly * 12
    std_annualized = std_monthly * np.sqrt(12)
    sharpe_ratio = mean_annualized / std_annualized if std_annualized != 0 else np.nan
    return sharpe_ratio

def evaluate_model(df_oos):
    """Aggregate R² across all OOS years."""
    oos_r2 = df_oos['oos_r2'].mean()
    naive_oos_r2 = df_oos['naive_oos_r2'].mean()
    return oos_r2, naive_oos_r2

def plot_results(df_oos):
    """Plot realized vs. predicted excess returns."""
    plt.figure(figsize=(10, 5))
    plt.scatter(df_oos['y_true'], df_oos['y_pred'], alpha=0.3)
    plt.axhline(0, color='grey', linestyle='--')
    plt.axvline(0, color='grey', linestyle='--')
    plt.title("OLS-3: Realized vs. Predicted Excess Returns")
    plt.xlabel("Realized")
    plt.ylabel("Predicted")
    plt.grid(True)
    plt.tight_layout()
    plt.savefig('ols3_custom_split_plot.png')
    plt.close()

# ----------------------------------------------------
# 3. Main Execution
# ----------------------------------------------------
def main():
    print(f"OLS-3 will use {len(FIRM_CHARS)} predictors")
    results = []
    all_ew_returns = []
    all_mcap_returns = []

    for _, row in ESTIMATION_PERIODS.iterrows():
        print(f"Running OLS-3 for OOS year {row['oos_year']}")
        train_df, oos_df = load_data(row)
        result, ew_returns, mcap_returns = train_model(train_df, oos_df, row['oos_year'])
        if result is not None:
            results.append(result)
            all_ew_returns.extend(ew_returns)
            all_mcap_returns.extend(mcap_returns)

    df_oos = pd.concat(results)
    oos_r2, naive_oos_r2 = evaluate_model(df_oos)

    # Calculate Sharpe Ratios
    ew_sharpe = calculate_sharpe_ratio(all_ew_returns)
    mcap_sharpe = calculate_sharpe_ratio(all_mcap_returns)

    print(f"Naive OOS R²: {naive_oos_r2:.4f}")
    print(f"OOS R²: {oos_r2:.4f}")
    print(f"Equal-Weighted Short-Long Sharpe Ratio: {ew_sharpe:.4f}")
    print(f"Market-Cap-Weighted Short-Long Sharpe Ratio: {mcap_sharpe:.4f}")

    plot_results(df_oos)

if __name__ == "__main__":
    main()